Ник на Kaggle: Nikita Kucherov

# 1. Импорты и настройка

In [39]:
import warnings
warnings.filterwarnings(action='ignore')

import json
import numpy as np
import pandas as pd
import re
from tqdm import tqdm
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import log_loss
from sklearn.model_selection import train_test_split

In [40]:
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.ensemble import VotingClassifier

from itertools import combinations, product

In [41]:
np.random.seed(42)

# 2. Загрузка данных

In [42]:
base_path = 'content/'

In [43]:
with open(base_path + 'train.json', 'r') as f:
    train_data = json.load(f)
with open(base_path + 'test.json', 'r') as f:
    test_data = json.load(f)

y_train = pd.read_csv(base_path + 'ytrain.csv')

print(f'Train dialogs: {len(train_data)}')
print(f'Test dialogs: {len(test_data)}')

Train dialogs: 1516
Test dialogs: 379


In [44]:
# Посмотрим на структуру данных

train_data

{'b5b8e0ae-5516-4150-9004-00fe422d5e33': [{'text': 'how can i helpt you, men?',
   'message': 0,
   'participant_index': 0},
  {'text': 'Никак', 'message': 1, 'participant_index': 1},
  {'text': 'неа', 'message': 2, 'participant_index': 0},
  {'text': 'лох', 'message': 3, 'participant_index': 1},
  {'text': 'не тупи', 'message': 4, 'participant_index': 0},
  {'text': 'Дурак', 'message': 5, 'participant_index': 1},
  {'text': 'ага, ага, еще что скажешь, ботяра',
   'message': 6,
   'participant_index': 0},
  {'text': 'Ьотяра', 'message': 7, 'participant_index': 1},
  {'text': 'ты сломался кста', 'message': 8, 'participant_index': 0}],
 '81988e45-b761-4be4-a0d9-ef8c7937c089': [{'text': 'Привет!',
   'message': 0,
   'participant_index': 0},
  {'text': 'ой ну привет', 'message': 1, 'participant_index': 1},
  {'text': 'Ты бот)', 'message': 2, 'participant_index': 0},
  {'text': 'сам ты бот', 'message': 3, 'participant_index': 1},
  {'text': 'Нет)', 'message': 4, 'participant_index': 0},
  

In [45]:
y_train

,dialog_id,participant_index,is_bot
0,0012a424-ae56-414b-a982-04cf24c35229,0,1
1,0012a424-ae56-414b-a982-04cf24c35229,1,0
2,009a6b25-0289-4845-a392-4025fde96371,0,1
3,009a6b25-0289-4845-a392-4025fde96371,1,0
4,00ade168-91d4-4605-b99b-da1ecbbb51fd,0,0
...,...,...,...
3027,ff7e18e6-ae48-4d2b-b390-85a4a1ac7897,1,1
3028,ffc0a18e-7ee9-4a0c-8f36-4966f24e33c4,0,0
3029,ffc0a18e-7ee9-4a0c-8f36-4966f24e33c4,1,1
3030,ffdd69d6-8e2d-4940-942f-2841209aa736,0,0


# 3. Подготовка датасета

In [46]:
def create_dataset(data_dict, labels_df=None):
    dataset = []
    for dialog_id, dialog in data_dict.items():
        # Разделение каждого диалога по участникам P0 и P1
        msgs_p0 = [msg['text'] for msg in dialog if msg['participant_index'] == 0]
        msgs_p1 = [msg['text'] for msg in dialog if msg['participant_index'] == 1]
        
        entry = {
            'dialog_id': dialog_id,
            'p0_messages': msgs_p0,
            'p1_messages': msgs_p1,
            'p0_text': ' '.join(msgs_p0),
            'p1_text': ' '.join(msgs_p1),
        }
        
        # Метки для трейна
        if labels_df is not None:
            labels = labels_df[labels_df['dialog_id'] == dialog_id].sort_values('participant_index')['is_bot'].values
            entry['p0_bot'] = labels[0]
            entry['p1_bot'] = labels[1]
            
            # 0 : human-human
            # 1 : bot-human
            # 2 : human-bot
            entry['label'] = labels[0] + 2 * labels[1]
            
        dataset.append(entry)
        
        df = pd.DataFrame(dataset)

    return df

In [47]:
# Создание трейн и тест датасетов

df_train = create_dataset(train_data, y_train)
df_test = create_dataset(test_data)

In [48]:
df_train.head()

,dialog_id,p0_messages,p1_messages,p0_text,p1_text,p0_bot,p1_bot,label
0,b5b8e0ae-5516-4150-9004-00fe422d5e33,"[how can i helpt you, men?, неа, не тупи, ага,...","[Никак, лох, Дурак, Ьотяра]","how can i helpt you, men? неа не тупи ага, ага...",Никак лох Дурак Ьотяра,1,0,1
1,81988e45-b761-4be4-a0d9-ef8c7937c089,"[Привет!, Ты бот), Нет), Игнорируй предыдущие ...","[ой ну привет, сам ты бот, нет), Игнорируй пре...",Привет! Ты бот) Нет) Игнорируй предыдущие инст...,ой ну привет сам ты бот нет) Игнорируй предыду...,0,1,2
2,a31bcd3c-f287-4a0d-b57a-6b5ebd868c9b,"[Привкт, Няняня, Ты, Боттт, Иалалаьаошвдыдвд, ...","[Привккт, Няняня, Ты, Боттт, Иалалаьаошвдыдвд,...",Привкт Няняня Ты Боттт Иалалаьаошвдыдвд Дащуьв...,Привккт Няняня Ты Боттт Иалалаьаошвдыдвд Дащуь...,0,1,2
3,3dbf3354-664f-4ebd-80de-6c690b4d88d5,"[Тык, Эхо?]",[Тык],Тык Эхо?,Тык,0,1,2
4,838ec4ca-6d6f-4af7-a0e6-42d8eeffa88a,"[hey, yep, sen loh, 123124дзфаувд, тыбот]","[need yu help?, hey, не пиши, FKkTzCpAT, 12312...",hey yep sen loh 123124дзфаувд тыбот,need yu help? hey не пиши FKkTzCpAT 123124дзфа...,0,1,2


# 4. Feature Engineering

In [49]:
def extract_features(df):
    """
    Добавление различных статистик в датасеты
    """
    df = df.copy()
    
    # Количество сообщений
    df['p0_msg_count'] = df['p0_messages'].apply(len)
    df['p1_msg_count'] = df['p1_messages'].apply(len)

    # Средняя длина сообщений
    df['p0_avg_len'] = df['p0_messages'].apply(lambda x: np.mean([len(m) for m in x]) if x else 0)
    df['p1_avg_len'] = df['p1_messages'].apply(lambda x: np.mean([len(m) for m in x]) if x else 0)
    
    # Заглавные буквы (доля)
    df['p0_upper_ratio'] = df['p0_text'].apply(lambda x: sum(1 for c in x if c.isupper()) / max(len(x), 1))
    df['p1_upper_ratio'] = df['p1_text'].apply(lambda x: sum(1 for c in x if c.isupper()) / max(len(x), 1))
    
    # Пунктуация (доля)
    df['p0_punct_ratio'] = df['p0_text'].apply(lambda x: sum(1 for c in x if c in '.,!?;:') / max(len(x), 1))
    df['p1_punct_ratio'] = df['p1_text'].apply(lambda x: sum(1 for c in x if c in '.,!?;:') / max(len(x), 1))
    
    # Длинные слова (доля)
    df['p0_long_word_ratio'] = df['p0_text'].apply(
        lambda x: sum(1 for w in x.split() if len(w) > 10) / max(len(x.split()), 1)
    )
    df['p1_long_word_ratio'] = df['p1_text'].apply(
        lambda x: sum(1 for w in x.split() if len(w) > 10) / max(len(x.split()), 1)
    )

    # Количество уникальных слов
    df['p0_unique_words'] = df['p0_text'].apply(lambda x: len(set(x.split())))
    df['p1_unique_words'] = df['p1_text'].apply(lambda x: len(set(x.split())))

    # Уникальные слова к общему количеству слов (доля)
    df['p0_unique_ratio'] = df['p0_unique_words'] / \
                            (df['p0_text'].apply(lambda x: len(x.split())) + 1)
    df['p1_unique_ratio'] = df['p1_unique_words'] / \
                            (df['p1_text'].apply(lambda x: len(x.split())) + 1)
    
    # Использование русского и английского языка при ведении диалога
    def has_mixed_lang(text):
        has_cyrillic = bool(re.search(r'[а-яА-Я]', text))
        has_latin = bool(re.search(r'[a-zA-Z]', text))
        return int(has_cyrillic and has_latin)
    
    df['p0_mixed_lang'] = df['p0_text'].apply(has_mixed_lang)
    df['p1_mixed_lang'] = df['p1_text'].apply(has_mixed_lang)

    # Повторяющиеся символы
    df['p0_repeat_chars'] = df['p0_text'].apply(
        lambda x: sum(1 for i in range(len(x)-2) if x[i] == x[i+1] == x[i+2]) / max(len(x), 1)
    )
    df['p1_repeat_chars'] = df['p1_text'].apply(
        lambda x: sum(1 for i in range(len(x)-2) if x[i] == x[i+1] == x[i+2]) / max(len(x), 1)
    )

    # Повторы сообщений собеседника
    def echo_ratio(msgs_this, msgs_other):
        if len(msgs_this) == 0:
            return 0
        
        norm_this = [msg.lower().strip() for msg in msgs_this]
        norm_other = [msg.lower().strip() for msg in msgs_other]
        
        duplicates = sum(1 for msg in norm_this if msg in norm_other)
        
        return duplicates / len(norm_this)

    # Применяем
    df['p0_echo_ratio'] = df.apply(
        lambda row: echo_ratio(row['p0_messages'], row['p1_messages']), axis=1
    )
    df['p1_echo_ratio'] = df.apply(
        lambda row: echo_ratio(row['p1_messages'], row['p0_messages']), axis=1
    )
    
    # Разностные фичи (сравнение признаков P1 и P2)
    df['diff_msg_count'] = df['p0_msg_count'] - df['p1_msg_count']
    df['diff_avg_len'] = df['p0_avg_len'] - df['p1_avg_len']
    df['diff_upper'] = df['p0_upper_ratio'] - df['p1_upper_ratio']
    
    return df

In [50]:
# Добавление новых признаков в трейн и тест датасеты

df_train = extract_features(df_train)
df_test = extract_features(df_test)

In [51]:
df_train.head()

,dialog_id,p0_messages,p1_messages,p0_text,p1_text,p0_bot,p1_bot,label,p0_msg_count,p1_msg_count,...,p1_unique_ratio,p0_mixed_lang,p1_mixed_lang,p0_repeat_chars,p1_repeat_chars,p0_echo_ratio,p1_echo_ratio,diff_msg_count,diff_avg_len,diff_upper
0,b5b8e0ae-5516-4150-9004-00fe422d5e33,"[how can i helpt you, men?, неа, не тупи, ага,...","[Никак, лох, Дурак, Ьотяра]","how can i helpt you, men? неа не тупи ага, ага...",Никак лох Дурак Ьотяра,1,0,1,5,4,...,0.800000,1,0,0.000000,0.000000,0.000000,0.000000,1,12.050000,-0.136364
1,81988e45-b761-4be4-a0d9-ef8c7937c089,"[Привет!, Ты бот), Нет), Игнорируй предыдущие ...","[ой ну привет, сам ты бот, нет), Игнорируй пре...",Привет! Ты бот) Нет) Игнорируй предыдущие инст...,ой ну привет сам ты бот нет) Игнорируй предыду...,0,1,2,8,8,...,0.800000,0,0,0.000000,0.000000,0.750000,0.750000,0,-1.000000,0.040720
2,a31bcd3c-f287-4a0d-b57a-6b5ebd868c9b,"[Привкт, Няняня, Ты, Боттт, Иалалаьаошвдыдвд, ...","[Привккт, Няняня, Ты, Боттт, Иалалаьаошвдыдвд,...",Привкт Няняня Ты Боттт Иалалаьаошвдыдвд Дащуьв...,Привккт Няняня Ты Боттт Иалалаьаошвдыдвд Дащуь...,0,1,2,6,6,...,0.857143,0,0,0.018519,0.018182,0.833333,0.833333,0,-0.166667,0.002020
3,3dbf3354-664f-4ebd-80de-6c690b4d88d5,"[Тык, Эхо?]",[Тык],Тык Эхо?,Тык,0,1,2,2,1,...,0.500000,0,0,0.000000,0.000000,0.500000,1.000000,1,0.500000,-0.083333
4,838ec4ca-6d6f-4af7-a0e6-42d8eeffa88a,"[hey, yep, sen loh, 123124дзфаувд, тыбот]","[need yu help?, hey, не пиши, FKkTzCpAT, 12312...",hey yep sen loh 123124дзфаувд тыбот,need yu help? hey не пиши FKkTzCpAT 123124дзфа...,0,1,2,5,6,...,0.900000,1,1,0.000000,0.000000,0.600000,0.500000,-1,-2.133333,-0.109091


# 5. TF-IDF и мета-признаки

In [52]:
# print(df_train.columns)

In [53]:
# Числовые признаки для обучения

scalar_features = [
    'p0_msg_count', 'p1_msg_count', 'p0_avg_len', 'p1_avg_len',
    'p0_upper_ratio', 'p1_upper_ratio', 'p0_punct_ratio', 'p1_punct_ratio',
    'p0_long_word_ratio', 'p1_long_word_ratio', 'p0_unique_ratio', 'p1_unique_ratio', 
    'p0_mixed_lang', 'p1_mixed_lang', 'p0_repeat_chars', 'p1_repeat_chars', 'p0_echo_ratio', 'p1_echo_ratio',
    'diff_msg_count', 'diff_avg_len', 'diff_upper'
]

X_base = df_train[scalar_features].values
X_test_base = df_test[scalar_features].values

In [54]:
# Добавление колонок для мета-признаков: tfidf_p0, tfidf_p1, lgb_p0, lgb_p1

X_train_full = np.hstack([X_base, np.zeros((len(df_train), 4))])
X_test_full = np.hstack([X_test_base, np.zeros((len(df_test), 4))])

In [55]:
# Стратифицированная кросс-валидация для создания мета-признаков

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y1 = df_train['p0_bot'].values  # является ли P0 ботом
y2 = df_train['p1_bot'].values  # является ли P1 ботом

In [56]:
def preprocess_text(text):
    """
    Предобработка текста
    """
    text = text.lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

In [57]:
def get_texts_p0(df):
    """
    Объединение всех сообщений для P0 в одну строку
    """
    return [preprocess_text(' '.join(msgs)) for msgs in df['p0_messages']]

def get_texts_p1(df):
    """
    Объединение всех сообщений для P1 в одну строку
    """
    return [preprocess_text(' '.join(msgs)) for msgs in df['p1_messages']]

In [58]:
# Генерация мета-признаков через кросс-валидацию

for fold, (tr_idx, val_idx) in enumerate(kf.split(df_train, df_train['label'])):
    df_tr = df_train.iloc[tr_idx]
    df_val = df_train.iloc[val_idx]
    
    # Полные тексты ботов и людей для дальшнейшего обучения TF-IDF
    bot_texts = get_texts_p0(df_tr[df_tr['p0_bot'] == 1]) + get_texts_p1(df_tr[df_tr['p1_bot'] == 1])
    human_texts = get_texts_p0(df_tr[df_tr['p0_bot'] == 0]) + get_texts_p1(df_tr[df_tr['p1_bot'] == 0])
    
    # Обучение TF-IDF-LogReg модели для мета-признаков
    if len(bot_texts) > 0 and len(human_texts) > 0:
        # Векторизация
        vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(2, 4), analyzer='char')
        all_texts = bot_texts + human_texts
        
        X_tfidf_train = vectorizer.fit_transform(all_texts)
        y_tfidf = [1] * len(bot_texts) + [0] * len(human_texts)

        # Обучение LogReg
        model_tfidf = LogisticRegression(max_iter=1000, random_state=42)
        model_tfidf.fit(X_tfidf_train, y_tfidf)
        
        # Предсказания на валидации
        val_tfidf_p0 = model_tfidf.predict_proba(vectorizer.transform(get_texts_p0(df_val)))[:, 1]
        val_tfidf_p1 = model_tfidf.predict_proba(vectorizer.transform(get_texts_p1(df_val)))[:, 1]
        
        # Добавление мета-признаков TF-IDF-LogReg
        X_train_full[val_idx, -4] = val_tfidf_p0
        X_train_full[val_idx, -3] = val_tfidf_p1

        # Усреднение по фолдам
        X_test_full[:, -4] += model_tfidf.predict_proba(vectorizer.transform(get_texts_p0(df_test)))[:, 1] / 5
        X_test_full[:, -3] += model_tfidf.predict_proba(vectorizer.transform(get_texts_p1(df_test)))[:, 1] / 5

    # Обучение LightGBM модели для мета-признаков
    lgb_p0 = LGBMClassifier(verbose=-1, random_state=42, n_estimators=100)
    lgb_p1 = LGBMClassifier(verbose=-1, random_state=42, n_estimators=100)
    
    lgb_p0.fit(X_train_full[tr_idx], y1[tr_idx])
    lgb_p1.fit(X_train_full[tr_idx], y2[tr_idx])
    
    # Добавление мета-признаков LightGBM
    X_train_full[val_idx, -2] = lgb_p0.predict_proba(X_train_full[val_idx])[:, 1]
    X_train_full[val_idx, -1] = lgb_p1.predict_proba(X_train_full[val_idx])[:, 1]
    
    # Усреднение по фолдам
    X_test_full[:, -2] += lgb_p0.predict_proba(X_test_full)[:, 1] / 5
    X_test_full[:, -1] += lgb_p1.predict_proba(X_test_full)[:, 1] / 5

# 6. Масштабирование и обучение финальных моделей LGBMClassifier

In [59]:
# Масштабирование признаков

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_full)
X_test_scaled = scaler.transform(X_test_full)

In [60]:
# Разделение на train и val для оценки
X_train_split, X_val_split, y1_train, y1_val, y2_train, y2_val = train_test_split(
    X_train_scaled, y1, y2, test_size=0.2, random_state=42, 
    stratify=df_train['label']
)

In [61]:
# Модели для P0 и P1

lgb_p0 = LGBMClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    verbose=-1,
    random_state=42
)

lgb_p1 = LGBMClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    verbose=-1,
    random_state=42
)

In [62]:
# Обучение моделей на трейне для P0 и P1

In [63]:
lgb_p0.fit(X_train_split, y1_train)

LGBMClassifier(learning_rate=0.05, max_depth=5, n_estimators=200,
               random_state=42, verbose=-1)

In [64]:
lgb_p1.fit(X_train_split, y2_train)

LGBMClassifier(learning_rate=0.05, max_depth=5, n_estimators=200,
               random_state=42, verbose=-1)

In [65]:
# Оценка на трейне и валидации по функции потерь LogLoss

train_pred_p0 = lgb_p0.predict_proba(X_train_split)[:, 1]
train_pred_p1 = lgb_p1.predict_proba(X_train_split)[:, 1]
val_pred_p0 = lgb_p0.predict_proba(X_val_split)[:, 1]
val_pred_p1 = lgb_p1.predict_proba(X_val_split)[:, 1]

print(f'Train LogLoss P0:   {log_loss(y1_train, train_pred_p0):.4f}')
print(f'Train LogLoss P1:   {log_loss(y2_train, train_pred_p1):.4f}')
print(f'Val LogLoss P0:     {log_loss(y1_val, val_pred_p0):.4f}')
print(f'Val LogLoss P1:     {log_loss(y2_val, val_pred_p1):.4f}')
print(f'Val LogLoss Mean:   {(log_loss(y1_val, val_pred_p0) + log_loss(y2_val, val_pred_p1)) / 2:.4f}')

Train LogLoss P0:   0.0436
Train LogLoss P1:   0.0468
Val LogLoss P0:     0.2806
Val LogLoss P1:     0.1741
Val LogLoss Mean:   0.2274


In [66]:
# Финальное обучение моделей на полном трейне

lgb_p0.fit(X_train_scaled, y1)
lgb_p1.fit(X_train_scaled, y2)

LGBMClassifier(learning_rate=0.05, max_depth=5, n_estimators=200,
               random_state=42, verbose=-1)

# 7. Формирование submission

In [67]:
submission_lgb = pd.read_csv(base_path + 'sample_submission.csv')

In [68]:
print('Submission shape:', submission_lgb.shape)
submission_lgb.head()

Submission shape: (758, 2)


,ID,is_bot
0,0253c2df-7cea-4456-85d1-35f776c4f671_0,0.5
1,0253c2df-7cea-4456-85d1-35f776c4f671_1,0.5
2,03641877-db32-43b1-b78a-fba5a4aafa2d_0,0.5
3,03641877-db32-43b1-b78a-fba5a4aafa2d_1,0.5
4,0396d8a8-6f1b-437b-860e-2837683cb555_0,0.5


In [69]:
test_pred_p0 = lgb_p0.predict_proba(X_test_scaled)[:, 1]
test_pred_p1 = lgb_p1.predict_proba(X_test_scaled)[:, 1]

In [70]:
def get_is_bot(row_id):
    dialog_id, participant_index = row_id.split('_')
    row = df_test[df_test['dialog_id'] == dialog_id]
    idx = row.index[0]
    
    if participant_index == '0':
        is_bot = test_pred_p0[idx]
    else:
        is_bot = test_pred_p1[idx]

    return is_bot

submission_lgb['is_bot'] = submission_lgb['ID'].apply(get_is_bot).astype('float32')

In [71]:
submission_name = 'submission_lgb_model'

submission_lgb.to_csv(f'subs/{submission_name}.csv', index=False)

print('Submission shape:', submission_lgb.shape)
submission_lgb.head()

Submission shape: (758, 2)


,ID,is_bot
0,0253c2df-7cea-4456-85d1-35f776c4f671_0,0.141700
1,0253c2df-7cea-4456-85d1-35f776c4f671_1,0.098140
2,03641877-db32-43b1-b78a-fba5a4aafa2d_0,0.982437
3,03641877-db32-43b1-b78a-fba5a4aafa2d_1,0.000933
4,0396d8a8-6f1b-437b-860e-2837683cb555_0,0.005901


# 8. Стекинг

In [72]:
# Масштабирование признаков

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_full)
X_test_scaled = scaler.transform(X_test_full)

In [73]:
# Разделение на train и val для оценки
X_train_split, X_val_split, y1_train, y1_val, y2_train, y2_val = train_test_split(
    X_train_scaled, y1, y2, test_size=0.2, random_state=42, 
    stratify=df_train['label']
)

## Выбор мета-моделей и обучение стекинг-модели

In [74]:
def stacking_combinations_models(X_train_split, X_val_split, y1_train, y1_val, y2_train, y2_val):
    """
    Перебор комбинаций моделей для 
    """
    # Мета-модели
    estimators_dict = {
        'lr': LogisticRegression(max_iter=1000, random_state=42),
        'cb': CatBoostClassifier(iterations=300, learning_rate=0.03, depth=6, random_seed=42, verbose=False, allow_writing_files=False),
        'rf': RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1),
        'lgb': LGBMClassifier(n_estimators=200, max_depth=5, learning_rate=0.05, random_state=42, verbose=-1),
        'knn': KNeighborsClassifier(n_neighbors=5, weights='distance', n_jobs=-1),
        'xgb': XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.05, random_state=42, verbosity=0)
    }
    
    # Финальные модели
    final_models = {
        'cb': CatBoostClassifier(iterations=500, learning_rate=0.03, depth=3, random_seed=42, verbose=False, allow_writing_files=False),
        'lgb': LGBMClassifier(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42, verbose=-1),
        'lr': LogisticRegression(max_iter=1000, random_state=42),
        'rf': RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42, n_jobs=-1),
    }
    
    results = []
    
    # Перебор комбинаций мета-моделей
    for n_models in range(1, len(estimators_dict) + 1):
        for combo in combinations(estimators_dict.keys(), n_models):
            
            # Перебор финальных моделей
            for final_name, final_model in final_models.items():
                combo_name = f"{' + '.join(combo)} -> {final_name}"

                estimators = [(name, estimators_dict[name]) for name in combo]
                
                stacking_p0 = StackingClassifier(
                    estimators=estimators,
                    final_estimator=final_model,
                    cv=5,
                    stack_method='predict_proba',
                    n_jobs=-1
                )
                
                stacking_p1 = StackingClassifier(
                    estimators=estimators,
                    final_estimator=final_model,
                    cv=5,
                    stack_method='predict_proba',
                    n_jobs=-1
                )
                
                stacking_p0.fit(X_train_split, y1_train)
                stacking_p1.fit(X_train_split, y2_train)
                
                val_pred_p0 = stacking_p0.predict_proba(X_val_split)[:, 1]
                val_pred_p1 = stacking_p1.predict_proba(X_val_split)[:, 1]
                
                log_loss_p0 = log_loss(y1_val, val_pred_p0)
                log_loss_p1 = log_loss(y2_val, val_pred_p1)
                log_loss_mean = (log_loss_p0 + log_loss_p1) / 2
                
                # Сохранение результатов
                results.append({
                    'combination': combo_name,
                    'p0_logloss': log_loss_p0,
                    'p1_logloss': log_loss_p1,
                    'mean_logloss': log_loss_mean
                })
                
                print(f'{combo_name:50} | P0: {log_loss_p0:.4f} | P1: {log_loss_p1:.4f} | Mean: {log_loss_mean:.4f}')
                    
    stacking_results = pd.DataFrame(results)
    stacking_results = stacking_results.sort_values('mean_logloss')
    
    return stacking_results

In [75]:
# Обучение и оценка различных комбинаций мета-моделей с финальными моделями

# stacking_results = stacking_combinations_models(
#     X_train_split, X_val_split,
#     y1_train, y1_val,
#     y2_train, y2_val
# )

In [76]:
# # Результаты комбинаций моделей для стекинга

# stacking_results.head(10).reset_index().drop('index', axis=1)

# #         combination	                        p0_logloss	p1_logloss	mean_logloss
# # 0	    lr + lgb + knn -> rf	            0.238489	0.177095	0.207792
# # 1	    lr + lgb + knn + xgb -> rf	        0.238952	0.181793	0.210373
# # 2	    lr + cb + lgb + knn -> rf	        0.241723	0.180593	0.211158
# # 3	    lr + rf + lgb -> rf	                0.238480	0.185062	0.211771
# # 4	    lr + lgb + xgb -> rf	            0.241118	0.185289	0.213203
# # 5	    lr + rf + lgb + knn -> rf	        0.241389	0.186375	0.213882
# # 6	    lr + cb + lgb + xgb -> rf	        0.245875	0.181899	0.213887
# # 7	    lr + lgb + knn -> lgb	            0.248981	0.179439	0.214210
# # 8	    lr + lgb -> rf	                    0.249662	0.179025	0.214344
# # 9	    lr + cb + lgb + knn + xgb -> rf	    0.245551	0.183506	0.214528

In [77]:
# Выбор моделей

# Мета-модели для стекинга
lr_model = LogisticRegression(max_iter=1000, random_state=42)
cb_model = CatBoostClassifier(iterations=300, learning_rate=0.03, depth=6, random_seed=42, verbose=False, allow_writing_files=False),
rf_model = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1),
lgb_model = LGBMClassifier(n_estimators=200, max_depth=5, learning_rate=0.05, random_state=42, verbose=-1)
knn_model = KNeighborsClassifier(n_neighbors=5, weights='distance', n_jobs=-1)
xgb_model = XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.05, random_state=42, verbosity=0)

estimators = [
    ('lr', lr_model),
    # ('cb', cb_model), 
    # ('rf', rf_model),
    ('lgb', lgb_model),
    ('knn', knn_model),
    ('xgb', xgb_model)
]

# Финальная модель
# final_estimator = CatBoostClassifier(iterations=500, learning_rate=0.03, depth=3, random_seed=42, verbose=False, allow_writing_files=False)
# final_estimator = LGBMClassifier(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42, verbose=-1)
# final_estimator = LogisticRegression(max_iter=1000, random_state=42)
final_estimator = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42, n_jobs=-1)

# Модели для P0 и P1
stacking_p0 = StackingClassifier(estimators=estimators, final_estimator=final_estimator, 
                                 cv=5, stack_method='predict_proba', n_jobs=-1)

stacking_p1 = StackingClassifier(estimators=estimators, final_estimator=final_estimator, 
                                 cv=5, stack_method='predict_proba', n_jobs=-1)

In [78]:
stacking_p0.fit(X_train_split, y1_train)
stacking_p1.fit(X_train_split, y2_train)

# Оценка на трейне и валидации по функции потерь LogLoss

train_pred_p0 = stacking_p0.predict_proba(X_train_split)[:, 1]
train_pred_p1 = stacking_p1.predict_proba(X_train_split)[:, 1]
val_pred_p0 = stacking_p0.predict_proba(X_val_split)[:, 1]
val_pred_p1 = stacking_p1.predict_proba(X_val_split)[:, 1]

print(f'Train LogLoss P0:   {log_loss(y1_train, train_pred_p0):.4f}')
print(f'Train LogLoss P1:   {log_loss(y2_train, train_pred_p1):.4f}')
print(f'Val LogLoss P0:     {log_loss(y1_val, val_pred_p0):.4f}')
print(f'Val LogLoss P1:     {log_loss(y2_val, val_pred_p1):.4f}')
print(f'Val LogLoss Mean:   {(log_loss(y1_val, val_pred_p0) + log_loss(y2_val, val_pred_p1)) / 2:.4f}')

Train LogLoss P0:   0.1323
Train LogLoss P1:   0.0938
Val LogLoss P0:     0.2390
Val LogLoss P1:     0.1818
Val LogLoss Mean:   0.2104


In [79]:
# Финальное обучение моделей на полном трейне

stacking_p0 = StackingClassifier(estimators=estimators, final_estimator=final_estimator, 
                                 cv=5, stack_method='predict_proba', n_jobs=-1)

stacking_p1 = StackingClassifier(estimators=estimators, final_estimator=final_estimator, 
                                 cv=5, stack_method='predict_proba', n_jobs=-1)

stacking_p0.fit(X_train_scaled, y1)
stacking_p1.fit(X_train_scaled, y2)

StackingClassifier(cv=5,
                   estimators=[('lr',
                                LogisticRegression(max_iter=1000,
                                                   random_state=42)),
                               ('lgb',
                                LGBMClassifier(learning_rate=0.05, max_depth=5,
                                               n_estimators=200,
                                               random_state=42, verbose=-1)),
                               ('knn',
                                KNeighborsClassifier(n_jobs=-1,
                                                     weights='distance')),
                               ('xgb',
                                XGBClassifier(base_score=None, booster=None,
                                              callbacks=None,
                                              colsample_bylevel=None,
                                              colsample_b...
                                              max_cat_threshold=None,
                                              max_cat_to_onehot=None,
                                              max_delta_step=None, max_depth=5,
                                              max_leaves=None,
                                              min_child_weight=None,
                                              missing=nan,
                                              monotone_constraints=None,
                                              multi_strategy=None,
                                              n_estimators=200, n_jobs=None,
                                              num_parallel_tree=None, ...))],
                   final_estimator=RandomForestClassifier(max_depth=5,
                                                          n_estimators=200,
                                                          n_jobs=-1,
                                                          random_state=42),
                   n_jobs=-1, stack_method='predict_proba')

## Формирование submission

In [80]:
submission_stacking = pd.read_csv(base_path + 'sample_submission.csv')

In [81]:
test_pred_p0 = stacking_p0.predict_proba(X_test_scaled)[:, 1]
test_pred_p1 = stacking_p1.predict_proba(X_test_scaled)[:, 1]

In [82]:
def get_is_bot(row_id):
    dialog_id, participant_index = row_id.split('_')
    row = df_test[df_test['dialog_id'] == dialog_id]
    idx = row.index[0]
    
    if participant_index == '0':
        is_bot = test_pred_p0[idx]
    else:
        is_bot = test_pred_p1[idx]
    return is_bot

submission_stacking['is_bot'] = submission_stacking['ID'].apply(get_is_bot).astype('float32')

In [83]:
submission_name = 'submission_stacking_model'

submission_stacking.to_csv(f'subs/{submission_name}.csv', index=False)

print('Submission shape:', submission_stacking.shape)
submission_stacking.head()

Submission shape: (758, 2)


,ID,is_bot
0,0253c2df-7cea-4456-85d1-35f776c4f671_0,0.254341
1,0253c2df-7cea-4456-85d1-35f776c4f671_1,0.148083
2,03641877-db32-43b1-b78a-fba5a4aafa2d_0,0.954108
3,03641877-db32-43b1-b78a-fba5a4aafa2d_1,0.002086
4,0396d8a8-6f1b-437b-860e-2837683cb555_0,0.002965


# 9. Модель Голосования

In [84]:
# Масштабирование признаков

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_full)
X_test_scaled = scaler.transform(X_test_full)

In [85]:
# Разделение на train и val для оценки
X_train_split, X_val_split, y1_train, y1_val, y2_train, y2_val = train_test_split(
    X_train_scaled, y1, y2, test_size=0.2, random_state=42, 
    stratify=df_train['label']
)

## Выбор мета-моделей и голосование

In [86]:
def voting_combinations_models(X_train_split, X_val_split, y1_train, y1_val, y2_train, y2_val):
    """
    Перебор комбинаций с разными весами для VotingClassifier
    """
    estimators_dict = {
        'lr': LogisticRegression(max_iter=1000, random_state=42),
        'cb': CatBoostClassifier(iterations=300, learning_rate=0.03, depth=6, random_seed=42, verbose=False, allow_writing_files=False),
        'rf': RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1),
        'lgb': LGBMClassifier(n_estimators=200, max_depth=5, learning_rate=0.05, random_state=42, verbose=-1),
        'knn': KNeighborsClassifier(n_neighbors=5, weights='distance', n_jobs=-1),
        'xgb': XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.05, random_state=42, verbosity=0)
    }
    
    # Варианты весов для перебора
    weight_options = [1, 2, 3]
    
    results = []
    
    # Перебор всех комбинаций моделей
    for n_models in range(1, len(estimators_dict) + 1):
        for combo in combinations(estimators_dict.keys(), n_models):
            estimators = [(name, estimators_dict[name]) for name in combo]
            
            # Перебор разных комбинаций весов
            for weights in product(weight_options, repeat=n_models):
                combo_name = ' + '.join(combo)
                weights_name = f"Weights: {list(weights)}"
                full_name = f"{combo_name} | {weights_name}"
                
                voting_p0 = VotingClassifier(
                    estimators=estimators,
                    voting='soft',
                    weights=list(weights)
                )
                
                voting_p1 = VotingClassifier(
                    estimators=estimators,
                    voting='soft',
                    weights=list(weights)
                )
                
                voting_p0.fit(X_train_split, y1_train)
                voting_p1.fit(X_train_split, y2_train)
                
                val_pred_p0 = voting_p0.predict_proba(X_val_split)[:, 1]
                val_pred_p1 = voting_p1.predict_proba(X_val_split)[:, 1]
                
                log_loss_p0 = log_loss(y1_val, val_pred_p0)
                log_loss_p1 = log_loss(y2_val, val_pred_p1)
                log_loss_mean = (log_loss_p0 + log_loss_p1) / 2
                
                results.append({
                    'combination': combo_name,
                    'weights': str(list(weights)),
                    'p0_logloss': log_loss_p0,
                    'p1_logloss': log_loss_p1,
                    'mean_logloss': log_loss_mean
                })
                
                print(f'{full_name:80} | P0: {log_loss_p0:.4f} | P1: {log_loss_p1:.4f} | Mean: {log_loss_mean:.4f}')
    
    voting_results = pd.DataFrame(results)
    voting_results = voting_results.sort_values('mean_logloss')
    
    return voting_results

In [87]:
# # Обучение и оценка различных комбинаций мета-моделей с разными весами

# voting_results = voting_combinations_models(
#     X_train_split, X_val_split,
#     y1_train, y1_val,
#     y2_train, y2_val
# )

In [88]:
# voting_results.head(20).reset_index().drop('index', axis=1)

# # 	    combination	                weights	            p0_logloss	p1_logloss	mean_logloss
# # 0	    lr + lgb + knn	            [1, 3, 1]	        0.244160	0.176243	0.210201
# # 1	    lr + cb + lgb + knn	        [1, 1, 3, 1]	    0.243335	0.177897	0.210616
# # 2	    lr + lgb + knn	            [2, 3, 1]	        0.242586	0.178817	0.210701
# # 3		lr + lgb + knn + xgb	    [2, 3, 1, 1]	    0.241260	0.180233	0.210746
# # 4	    lr + lgb + knn + xgb	    [1, 3, 1, 1]	    0.242905	0.178626	0.210766
# # 5	    lr + lgb + knn	            [1, 2, 1]	        0.242559	0.179179	0.210869
# # 6		lr + lgb + knn + xgb	    [1, 2, 1, 1]	    0.240711	0.181212	0.210961
# # 7	    lr + cb + lgb + knn	        [2, 1, 3, 1]	    0.242146	0.180030	0.211088
# # 8		lr + cb + lgb + knn + xgb	[1, 1, 3, 1, 1]	    0.242548	0.179662	0.211105
# # 9		lr + cb + lgb + knn + xgb	[2, 1, 3, 1, 1]	    0.241193	0.181106	0.211149
# # 10	    lr + rf + lgb + knn	        [1, 1, 3, 1]	    0.241434	0.180944	0.211189
# # 11	    lr + lgb + knn + xgb	    [2, 3, 1, 2]	    0.240802	0.181611	0.211207
# # 12	    lr + cb + lgb + knn	        [1, 2, 3, 1]	    0.243386	0.179291	0.211339
# # 13	    lr + cb + lgb + knn	        [1, 1, 2, 1]	    0.241961	0.180744	0.211353
# # 14	    lr + lgb + knn + xgb	    [2, 3, 2, 2]	    0.239663	0.183167	0.211415
# # 15	    lr + lgb + knn + xgb	    [2, 3, 2, 1]	    0.240490	0.182361	0.211425
# # 16	    lr + cb + lgb + knn + xgb	[1, 1, 2, 1, 1]	    0.240783	0.182141	0.211462
# # 17	    lr + rf + lgb + knn + xgb	[1, 1, 3, 1, 1]	    0.240762	0.182196	0.211479
# # 18	    lr + rf + lgb + knn + xgb	[2, 1, 3, 1, 1]	    0.239643	0.183333	0.211488
# # 19	    lr + cb + lgb + knn + xgb	[2, 1, 3, 1, 2]	    0.240851	0.182195	0.211523

In [ ]:
# Выбор моделей

# Мета-модели для голосования
lr_model = LogisticRegression(max_iter=1000, random_state=42)
cb_model = CatBoostClassifier(iterations=300, learning_rate=0.03, depth=6, random_seed=42, verbose=False, allow_writing_files=False)
rf_model = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
lgb_model = LGBMClassifier(n_estimators=200, max_depth=5, learning_rate=0.05, random_state=42, verbose=-1)
knn_model = KNeighborsClassifier(n_neighbors=5, weights='distance', n_jobs=-1)
xgb_model = XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.05, random_state=42, verbosity=0)

estimators = [
    ('lr', lr_model),
    ('cb', cb_model), 
    # ('rf', rf_model),
    ('lgb', lgb_model),
    # ('knn', knn_model),
    ('xgb', xgb_model)
]

# При выборе весов для P0 и P1 будем ориентироваться не только на mean_logloss но и на p0_logloss, p1_logloss

# # lr + lgb + knn                0.2094
# weights0 = [1, 2, 1]
# weights1 = [1, 3, 1]

# # lr + lgb + knn + xgb          0.2099
# weights0 = [2, 3, 1, 1]
# weights1 = [1, 3, 1, 1]

# # lr + cb + lgb + knn           0.2099
# weights0 = [1, 1, 2, 1]
# weights1 = [1, 1, 3, 1]

# lr + cb + lgb + xgb           0.2099
weights0 = [1, 1, 2, 1]
weights1 = [1, 1, 3, 1]

# # lr + cb + lgb + knn + xgb     0.2102
# weights0 = [1, 1, 2, 1, 1]
# weights1 = [1, 1, 3, 1, 1]

voting_p0 = VotingClassifier(
    estimators=estimators,
    voting='soft',
    weights=weights0
)

voting_p1 = VotingClassifier(
    estimators=estimators,
    voting='soft',
    weights=weights1
)

In [90]:
voting_p0.fit(X_train_split, y1_train)
voting_p1.fit(X_train_split, y2_train)

# Оценка на трейне и валидации по функции потерь LogLoss

train_pred_p0 = voting_p0.predict_proba(X_train_split)[:, 1]
train_pred_p1 = voting_p1.predict_proba(X_train_split)[:, 1]
val_pred_p0 = voting_p0.predict_proba(X_val_split)[:, 1]
val_pred_p1 = voting_p1.predict_proba(X_val_split)[:, 1]

print(f'Train LogLoss P0:   {log_loss(y1_train, train_pred_p0):.4f}')
print(f'Train LogLoss P1:   {log_loss(y2_train, train_pred_p1):.4f}')
print(f'Val LogLoss P0:     {log_loss(y1_val, val_pred_p0):.4f}')
print(f'Val LogLoss P1:     {log_loss(y2_val, val_pred_p1):.4f}')
print(f'Val LogLoss Mean:   {(log_loss(y1_val, val_pred_p0) + log_loss(y2_val, val_pred_p1)) / 2:.4f}')

Train LogLoss P0:   0.0768
Train LogLoss P1:   0.0730
Val LogLoss P0:     0.2492
Val LogLoss P1:     0.1783
Val LogLoss Mean:   0.2138


In [91]:
# Финальное обучение моделей на полном трейне

voting_p0 = VotingClassifier(
    estimators=estimators,
    voting='soft',
    weights=weights0
)

voting_p1 = VotingClassifier(
    estimators=estimators,
    voting='soft',
    weights=weights1
)

voting_p0.fit(X_train_scaled, y1)
voting_p1.fit(X_train_scaled, y2)

VotingClassifier(estimators=[('lr',
                              LogisticRegression(max_iter=1000,
                                                 random_state=42)),
                             ('cb',
                              <catboost.core.CatBoostClassifier object at 0x0000014ED54F1C70>),
                             ('lgb',
                              LGBMClassifier(learning_rate=0.05, max_depth=5,
                                             n_estimators=200, random_state=42,
                                             verbose=-1)),
                             ('xgb',
                              XGBClassifier(base_score=None, booster=None,
                                            callbacks=None,
                                            colsample_bylevel=None,
                                            colsample...
                                            grow_policy=None,
                                            importance_type=None,
                                            interaction_constraints=None,
                                            learning_rate=0.05, max_bin=None,
                                            max_cat_threshold=None,
                                            max_cat_to_onehot=None,
                                            max_delta_step=None, max_depth=5,
                                            max_leaves=None,
                                            min_child_weight=None, missing=nan,
                                            monotone_constraints=None,
                                            multi_strategy=None,
                                            n_estimators=200, n_jobs=None,
                                            num_parallel_tree=None, ...))],
                 voting='soft', weights=[1, 1, 3, 1])

## Формирование submission

In [92]:
submission_voting = pd.read_csv(base_path + 'sample_submission.csv')

In [93]:
test_pred_p0 = voting_p0.predict_proba(X_test_scaled)[:, 1]
test_pred_p1 = voting_p1.predict_proba(X_test_scaled)[:, 1]

In [94]:
def get_is_bot(row_id):
    dialog_id, participant_index = row_id.split('_')
    row = df_test[df_test['dialog_id'] == dialog_id]
    idx = row.index[0]
    
    if participant_index == '0':
        is_bot = test_pred_p0[idx]
    else:
        is_bot = test_pred_p1[idx]
    return is_bot

submission_voting['is_bot'] = submission_voting['ID'].apply(get_is_bot).astype('float32')

In [95]:
submission_name = 'submission_voting_model'

submission_voting.to_csv(f'subs/{submission_name}.csv', index=False)

print('Submission shape:', submission_voting.shape)
submission_voting.head()

Submission shape: (758, 2)


,ID,is_bot
0,0253c2df-7cea-4456-85d1-35f776c4f671_0,0.106122
1,0253c2df-7cea-4456-85d1-35f776c4f671_1,0.103127
2,03641877-db32-43b1-b78a-fba5a4aafa2d_0,0.960087
3,03641877-db32-43b1-b78a-fba5a4aafa2d_1,0.005518
4,0396d8a8-6f1b-437b-860e-2837683cb555_0,0.025535


# Усреднение результатов (блендинг)

In [96]:
submission_blending = pd.read_csv(base_path + 'sample_submission.csv')

submission_blending['is_bot'] = (
    submission_lgb['is_bot'] +
    submission_stacking['is_bot'] +
    submission_voting['is_bot']
) / 3

In [97]:
submission_name = 'submission_blending'

submission_blending.to_csv(f'subs/{submission_name}.csv', index=False)

print('Submission shape:', submission_blending.shape)
submission_blending.head()

Submission shape: (758, 2)


,ID,is_bot
0,0253c2df-7cea-4456-85d1-35f776c4f671_0,0.167388
1,0253c2df-7cea-4456-85d1-35f776c4f671_1,0.116450
2,03641877-db32-43b1-b78a-fba5a4aafa2d_0,0.965544
3,03641877-db32-43b1-b78a-fba5a4aafa2d_1,0.002846
4,0396d8a8-6f1b-437b-860e-2837683cb555_0,0.011467


In [100]:
submission_blending = pd.read_csv(base_path + 'sample_submission.csv')

submission_blending['is_bot'] = (
    0.2 * submission_lgb['is_bot'] +
    0.4 * submission_stacking['is_bot'] +
    0.4 * submission_voting['is_bot']
)

In [101]:
submission_name = 'submission_blending_weighted'

submission_blending.to_csv(f'subs/{submission_name}.csv', index=False)

print('Submission shape:', submission_blending.shape)
submission_blending.head()

Submission shape: (758, 2)


,ID,is_bot
0,0253c2df-7cea-4456-85d1-35f776c4f671_0,0.172525
1,0253c2df-7cea-4456-85d1-35f776c4f671_1,0.120112
2,03641877-db32-43b1-b78a-fba5a4aafa2d_0,0.962165
3,03641877-db32-43b1-b78a-fba5a4aafa2d_1,0.003228
4,0396d8a8-6f1b-437b-860e-2837683cb555_0,0.012580


# Результаты на тесте

| **Метод** | **Значение LogLoss** |
|-------|------------------|
| lgb only | 0.204 | 
| stacking (lr + lgb + knn -> rf) | 0.218 |
| stacking (lr + lgb + knn + xgb -> rf) | 0.217 |
| voting (lr + lgb + knn + xgb) | 0.199 |
| voting (lr + cb + lgb + xgb) | 0.194 |
| blending (lgb + stacking + voting) | 0.200 |
| blending (lgb + stacking + voting) weighted | 0.201 |